In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

from anngeno import AnnGeno
from scripts import get_burdens
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Get missense variants for required gene

In [ ]:
pg = pl.read_parquet('/home/dnanexus/data_dir/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').filter(pl.col('gene_name').is_in(['BRCA1']))
pg

In [ ]:
pg['file_name'].value_counts().sort('count', descending=True)

In [ ]:
plt.hist(pg['dms_score'], bins=100)
plt.show()

In [ ]:
anno = pl.read_parquet('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag/annotations.parquet').with_columns(
    pl.when(
        pl.col('amino_acids').is_not_null() & pl.col('protein_position').is_not_null()
    ).then(
        pl.col('amino_acids').str.split('/').list.get(0) +
        pl.col('protein_position').str.split('/').list.get(0) +
        pl.col('amino_acids').str.split('/').list.get(1)
    ).otherwise(None).alias('mutant')
).filter(
    # Filter for BRCA1 gene
    pl.col('region').is_in(['ENSG00000012048'])
)

id_cols = ['chrom', 'pos', 'ref', 'alt', 'id', 'region', 'col', 'AF_ukb', 'mutant', 'amino_acids', 'protein_position', 'consequence']
missense_annos = ['loftee_hc', 'CADD_RAW', 'am_pathogenicity', 'Consequence_missense_variant', 'PolyPhen', 'CADD_SIFTval', 'CADD_priPhCons', 'CADD_mamPhCons', 'CADD_verPhCons', 'gpn_score']

anno = anno.select(id_cols + missense_annos)
anno

In [ ]:
anno.filter(pl.col('Consequence_missense_variant')==1)

In [ ]:
pg.filter(~pl.col('mutant').is_in(anno['mutant']))

In [ ]:
brca_df = pg.filter(pl.col('file_name').is_in(['BRCA1_HUMAN_Findlay_2018']))[['mutant', 'dms_score', 'gene_name', 'file_name', 'region']].join(anno.filter(pl.col('region') == 'ENSG00000012048'), on=['region', 'mutant'], how='inner')

annos2compare = missense_annos + ['dms_score']
brca_df

### Check how exp. scores correlate with comp. scores

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='am_pathogenicity')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='Alphamissense') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='CADD_RAW')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='CADD RAW') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='gpn_score')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='GPN-MSA') +
    theme_bw()
)

### Pos-neg split scores

In [ ]:

def check_positive_negative(df: pl.DataFrame, columns):
    results = {}
    for col in columns:
        if col in df.columns:
            non_null = df.select(pl.col(col).drop_nulls())[col]
            if non_null.is_empty():
                results[col] = False  # Only nulls
            else:
                min_val = non_null.min()
                max_val = non_null.max()
                results[col] = (min_val < 0) and (max_val > 0)
        else:
            results[col] = False  # Column not found
    return results

# Example usage:
positive_negative_check = check_positive_negative(brca_df, annos2compare)

# Print the results
for column, has_both in positive_negative_check.items():
    if has_both:
        print(f"Column '{column}': Contains both positive and negative values.")

In [ ]:
def split_pos_neg_lazy(df: pl.LazyFrame, columns):
    # Start with the lazy frame
    lf = df

    for col in columns:
        if col in df.columns:
            pos_col = (
                pl.when(pl.col(col) > 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_pos")
            )

            neg_col = (
                pl.when(pl.col(col) < 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_neg")
            )

            lf = lf.with_columns([pos_col, neg_col])

    return lf

# Get the columns that contain both positive and negative values
mix_cols = [k for k, v in positive_negative_check.items() if v]

# Split the positive and negative values into separate columns
split_ann = split_pos_neg_lazy(brca_df.lazy(), mix_cols).collect()
split_ann

## Compute burdens

In [ ]:
split_ann

In [ ]:
type(split_ann.lazy())

In [ ]:
ag = AnnGeno(filename='/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag', filemode="r", low_mem=True)
ag

In [ ]:
ag._set_annotations(split_ann.lazy())

In [ ]:
ag.get_many_regions(['ENSG00000012048'])

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_dms.yaml"
output_dir = "/home/dnanexus/data_dir/dms_burdens/"

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    gene_list=['ENSG00000012048'],
    output_dir=output_dir,
    only_snps=True,
    na_mask=True,
    overwrite=True,
    gene_chunk_size=1,
    sample_chunk_size=50_000,
    new_annotation_df=split_ann.lazy(),
    variant_subset=split_ann['id'].unique().to_list()
)